In [1]:
import sys
from pathlib import Path

# locate project root
root = Path.cwd().resolve()
while root != root.parent and not (root / "shared").exists():
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.paths import get_experiment_paths
from shared.visualization import show_grid, save_grid
from shared.weight_init import dcgan_weights_init
from shared.seed import seed_everything

seed_everything(42)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

paths = get_experiment_paths("05_dcgan_fashionmnist")

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.FashionMNIST(
    root=str(paths.datasets_dir),
    train=True,
    download=True,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

len(dataset)

100%|██████████| 26.4M/26.4M [00:05<00:00, 4.56MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 129kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.85MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


60000

In [3]:
import torch.nn as nn

z_dim = 100
feat = 64

class Generator(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.ConvTranspose2d(z_dim, feat*4, 7, 1, 0, bias=False),
            nn.BatchNorm2d(feat*4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*4, feat*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat*2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*2, feat, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat),
            nn.ReLU(True),

            nn.Conv2d(feat, 1, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

In [4]:
class Discriminator(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(1, feat, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat, feat*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat*2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*2, feat*4, 3, 2, 1, bias=False),
            nn.BatchNorm2d(feat*4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*4, 1, 4, 1, 0)
        )

    def forward(self, x):
        out = self.net(x)
        return out.view(-1)

In [5]:
import torch.optim as optim

G = Generator().to(device)
D = Discriminator().to(device)

G.apply(dcgan_weights_init)
D.apply(dcgan_weights_init)

criterion = nn.BCEWithLogitsLoss()

g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
d_opt = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))

fixed_z = torch.randn(64, z_dim, 1, 1, device=device)

In [ ]:
epochs = 30

for epoch in range(1, epochs+1):

    for real,_ in loader:

        real = real.to(device)
        b = real.size(0)

        # Train D
        z = torch.randn(b, z_dim,1,1, device=device)
        fake = G(z).detach()

        d_real = D(real)
        d_fake = D(fake)

        real_labels = torch.ones(b, device=device)*0.9
        fake_labels = torch.zeros(b, device=device)

        d_loss = (criterion(d_real, real_labels) +
                  criterion(d_fake, fake_labels)) / 2

        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        # Train G
        z = torch.randn(b, z_dim,1,1, device=device)
        fake = G(z)

        d_fake = D(fake)

        g_loss = criterion(d_fake, torch.ones(b, device=device))

        g_opt.zero_grad()
        g_loss.backward()
        g_opt.step()

    G.eval()
    with torch.no_grad():
        samples = G(fixed_z)

    save_grid(samples,
        paths.samples_dir / f"epoch_{epoch:03d}.png",
        nrow=8)

    torch.save({
        "epoch": epoch,
        "G": G.state_dict(),
        "D": D.state_dict(),
        "z_dim": z_dim
    }, paths.models_dir / f"ckpt_epoch_{epoch:03d}.pth")

    print(f"Epoch {epoch}/{epochs}  d_loss:{d_loss:.3f}  g_loss:{g_loss:.3f}")

Epoch 1/30  d_loss:0.422  g_loss:1.545
Epoch 2/30  d_loss:0.485  g_loss:1.625
Epoch 3/30  d_loss:0.588  g_loss:1.977
Epoch 4/30  d_loss:0.545  g_loss:2.160
Epoch 5/30  d_loss:0.556  g_loss:0.954
Epoch 6/30  d_loss:0.552  g_loss:1.103
Epoch 7/30  d_loss:0.560  g_loss:1.388
Epoch 8/30  d_loss:0.533  g_loss:0.821
Epoch 9/30  d_loss:0.493  g_loss:1.495
Epoch 10/30  d_loss:0.479  g_loss:1.433
Epoch 11/30  d_loss:0.526  g_loss:0.895
Epoch 12/30  d_loss:0.667  g_loss:0.764
Epoch 13/30  d_loss:0.539  g_loss:1.777
Epoch 14/30  d_loss:0.482  g_loss:1.024
Epoch 15/30  d_loss:0.495  g_loss:1.989
Epoch 16/30  d_loss:0.570  g_loss:2.255
Epoch 17/30  d_loss:0.520  g_loss:1.425
Epoch 18/30  d_loss:0.484  g_loss:1.453
Epoch 19/30  d_loss:0.471  g_loss:1.531
Epoch 20/30  d_loss:0.497  g_loss:1.181
Epoch 21/30  d_loss:0.577  g_loss:1.757
Epoch 22/30  d_loss:0.409  g_loss:1.597
Epoch 23/30  d_loss:0.511  g_loss:1.282
Epoch 24/30  d_loss:0.465  g_loss:1.250
Epoch 25/30  d_loss:0.482  g_loss:1.627
Epoch 26/